In [ ]:
import decode
import decode.utils
import decode.neuralfitter.train.live_engine

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"DECODE version: {decode.utils.bookkeeping.decode_state()}")

In [ ]:
device = 'cuda:0'  # or 'cpu'
device_ix = 1  # possibly change device index (only for cuda)
threads = 4  #  number of threads, useful for CPU heavy computation. Change if you know what you are doing.
worker = 4  # number of workers for data loading. Change only if you know what you are doing.

torch.set_num_threads(threads)  # set num threads

if device != 'cpu':
    if (not torch.cuda.is_available()) or (not decode.simulation.psf_kernel.CubicSplinePSF.cuda_is_available()):
        raise ValueError("You have selected a non CPU device, but CUDA is not available."
                         "Refer to CPU version or check your installation.")

In [ ]:
torch.cuda.is_available(), decode.simulation.psf_kernel.CubicSplinePSF.cuda_is_available()

In [ ]:
for i in range(torch.cuda.device_count()):
   print(torch.cuda.get_device_properties(i).name)

In [ ]:
# uncomment and exectue only if you have not executed the example download cell above
calib_file = 'z_calib_like_star_no_abber_3dcal.mat'

In [ ]:
# copy default parameter files which are then changed as specified
decode.utils.param_io.copy_reference_param('')  # saved in current dir

In [ ]:
param = decode.utils.param_io.load_params('param_star.yaml')  # change path if you load custom file

param.Hardware.device = device
param.Hardware.device_ix = device_ix
param.Hardware.device_simulation = device
param.Hardware.torch_threads = threads
param.Hardware.num_worker_train = worker

param.InOut.calibration_file = calib_file
param.InOut.experiment_out = ''
display(param.InOut.to_dict())

# param.Camera.baseline = 50
# param.Camera.e_per_adu = 1.0
# param.Camera.em_gain = 1
# param.Camera.px_size =[59.0, 59.0] # Pixel Size in nano meter
# param.Camera.qe = 1.0                # Quantum efficiency
# param.Camera.read_sigma = 70.0
# param.Camera.spur_noise = 0.000
# param.Camera.photon_units = False

display(param.Camera.to_dict())

# param.Simulation.bg_uniform = [200.0, 400.0]           # background range to sample from. You can also specify a const. value as 'bg_uniform = 100'
# param.Simulation.emitter_av = 50                 # Average number of emitters per frame
# param.Simulation.emitter_extent[2] = [-5, 5]    # Volume in which emitters are sampled. x,y values should not be changed. z-range (in nm) should be adjusted according to the PSF
# param.Simulation.intensity_mu_sig = [2_00_000.0, int(2_00_000**0.5)]  # Average intensity and its standard deviation
# param.Simulation.lifetime_avg = 1.25                     # Average lifetime of each emitter in frames. A value between 1 and 2 works for most experiments

display(param.Simulation.to_dict())

In [ ]:
simulator, sim_test = decode.neuralfitter.train.live_engine.setup_random_simulation(param)
camera = decode.simulation.camera.Photon2Camera.parse(param)
param = decode.utils.param_io.autoset_scaling(param)


In [ ]:
if device != 'cpu':
    mem_gb = torch.cuda.get_device_properties(device).total_memory / 1e9
    print(f"Your approximate total GPU memory size on the set device {device} is {mem_gb:.2f} GB.")

param.HyperParameter.batch_size = 32

Once you are happy with you settings you can write the parameters to a file (or edit the param_friendly.yaml directly)

In [ ]:
param_out_path = 'notebook_training_gauss_v3.yaml' # or an alternative path
decode.utils.param_io.save_params(param_out_path, param)

Call
`python -m decode.neuralfitter.train.live_engine -p notebook_training_gauss_v3.yaml  # change path if you modified it`

